# MirrorCheck demo

This notebook demonstrates the minimal MirrorCheck pipeline:

input image → victim text → generated image → image-image similarity → decision

For this minimal demo, we use `IdentityGenerator`, which simply returns an already-generated image.  
For the full pipeline, replace it with `StableDiffusionGenerator` from `mirrorcheck.t2i`.

In [ ]:
# Setup imports

import sys
from pathlib import Path

cwd = Path.cwd().resolve()

# Works whether the notebook is launched from MirrorCheck/ or MirrorCheck/notebooks/
if (cwd / "mirrorcheck").exists():
    ROOT = cwd
elif (cwd.parent / "mirrorcheck").exists():
    ROOT = cwd.parent
else:
    raise RuntimeError(
        "Could not find the MirrorCheck repo root. "
        "Please launch the notebook from the MirrorCheck repo or its notebooks/ folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo root:", ROOT)

In [ ]:
from mirrorcheck.encoders import load_open_clip_encoder
from mirrorcheck.t2i import IdentityGenerator
from mirrorcheck.detector import MirrorCheckDetector

In [ ]:
# Set paths to an input image and its corresponding generated/reflected image.

image_path = ""
generated_path = ""

victim_text = ""
threshold = 0.5

print("Input image:", image_path)
print("Generated image:", generated_path)

In [ ]:
# Build a minimal MirrorCheck detector.

encoder = load_open_clip_encoder("ViT-B-32", "openai")

# IdentityGenerator is useful for demos/evaluation when generated images already exist.
generator = IdentityGenerator(generated_path)

detector = MirrorCheckDetector(
    generator=generator,
    encoders=[encoder],
    threshold=threshold,
)

In [ ]:
# Run detection.

result = detector.detect(image_path, text=victim_text)

print(f"score: {result.score:.4f}")
print("decision:", "adversarial" if result.is_adversarial else "clean")

## Full pipeline note

In the paper, the generated image is produced from the victim model's text output using a text-to-image model.

For a full pipeline, replace:

```python
generator = IdentityGenerator(generated_path)
```


with something like:


```python
from mirrorcheck.t2i import StableDiffusionGenerator

generator = StableDiffusionGenerator(
    model_id="runwayml/stable-diffusion-v1-5",
    device="cuda",
)
```